# Learned control-response memory for v008

Definitions only. Execute the code cells of `research/sequence_components_v005.ipynb` first. No data reads, target access, training, remote actions, or submission occurs on definition.

The approved arm retains the original compact scalar LSTM and its head, then adds eight learned nonlinear control-response channels, each with a positive learned exponential time constant. The response nonlinearity sees only standardized known `R`, `C`, and `u_in`; known `dt` sets physical elapsed-time decay. These are predictive latent responses, not a claim that valve opening measures airflow. Every breath starts from its own first response; no state survives a call or crosses breaths.

`v8_memory_model(spec, n_features, preprocessing, common_base_state=None)` accepts the existing training-only `fit_preprocessor` dictionary: `feature_names`, `x_mean`, and `x_std`. Name lookup selects the four allowed channels; scaler and index buffers enter `state_dict`. Pass the common baseline state via `common_base_state`, or call `load_common_base_state` immediately after construction. Existing encoder/head key names and shapes remain exact. The residual output is initialized to zero, so the common initialization predicts exactly the scalar baseline. Upstream memory gradients become active after the first residual-output update.

The stable affine parallel scan uses seven tensor stages for 80 samples. It avoids products in denominators, overflow-prone inverse exponentials, and CUDA cumsum/cumprod restrictions under Torch 2.2 strict determinism. No caller setting is weakened. Added trainable parameters: 201. Both the common initialization and all added state are serialized by the common trainer.

`v8_memory_selfcheck(device)` exercises synthetic breath isolation/reset, causal known-control-only access, exact baseline initialization and state roundtrip, positive/extreme time constants, analytic irregular elapsed-time decay, stable scan equivalence, and finite nonzero parameter gradients. Run the common v008 trainer's actual-GPU batch256 forward/backward/inference, exact-resume, <2x baseline cost, and 11GB memory gates before production. CPU checks do not establish CUDA runtime support or capacity.


In [ ]:
# Definitions only. Execute the v005 sequence engine first.
_V8_MEMORY_REQUIRED = ('V5PressureRNN', 'canonical_model_spec', '_sequence_device',
    'seed_sequence', '_rng_state', '_restore_rng')
if any(name not in globals() for name in _V8_MEMORY_REQUIRED):
    raise RuntimeError('Execute research/sequence_components_v005.ipynb first')

V8_MEMORY_VERSION = 'v008-learned-response-memory-v1'
V8_MEMORY_CHANNELS = 8
V8_MEMORY_SOURCES = ('R', 'C', 'u_in', 'dt')
V8_MEMORY_INITIAL_TAU_SECONDS = (.03, .05, .10, .20, .40, .80, 1.50, 2.50)
V8_MEMORY_TAU_FLOOR_SECONDS = 1e-4


def _v8_memory_context(preprocessing, n_features):
    """Select only known control scalers; never inspect target statistics."""
    names = preprocessing.get('feature_names')
    if names is None or len(names) != int(n_features) or len(set(names)) != len(names):
        raise ValueError('Memory requires unique training-preprocessor feature_names')
    names = tuple(names)
    if any('pressure' in name.lower() for name in names):
        raise ValueError('Pressure-derived inputs are not allowed')
    if any(name not in names for name in V8_MEMORY_SOURCES):
        raise ValueError('Memory requires known input channels R, C, u_in and dt')
    mean = np.asarray(preprocessing['x_mean'], dtype=np.float32)
    std = np.asarray(preprocessing['x_std'], dtype=np.float32)
    if mean.shape != (int(n_features),) or std.shape != mean.shape:
        raise ValueError('Training feature scaler dimension mismatch')
    if not np.isfinite(mean).all() or not np.isfinite(std).all() or (std <= 0).any():
        raise ValueError('Invalid training feature scaler')
    indices = np.asarray([names.index(name) for name in V8_MEMORY_SOURCES], dtype=np.int64)
    return names, indices, mean[indices].copy(), std[indices].copy()


def _v8_response_memory_scan(response, dt, tau):
    """Stable inclusive scan of h[t]=a[t]*h[t-1]+(1-a[t])*response[t].

    h[0]=response[0] resets every breath. Associative composition of affine
    maps takes ceil(log2(T)) tensor stages, without cumsum/cumprod, division by
    a decaying product, in-place autograd writes, or a Python step per sample.
    All retention factors are in [0,1], including extreme positive tau.
    """
    if response.ndim != 3 or dt.shape != response.shape[:2]:
        raise ValueError('Expected response[B,T,K] and elapsed intervals[B,T]')
    if response.shape[1] < 1 or tau.shape != (response.shape[-1],):
        raise ValueError('Nonempty sequence and one positive time constant per channel required')
    # Feature construction guarantees nonnegative physical dt. This clamp is
    # only a guard against float32 reconstruction roundoff at a zero interval.
    exponent = -dt.clamp_min(0).unsqueeze(-1) / tau.reshape(1, 1, -1)
    retention = torch.exp(exponent)
    update = -torch.expm1(exponent)
    # The first state depends on this breath only; ignore its nominal dt.
    a = torch.cat((torch.zeros_like(retention[:, :1]), retention[:, 1:]), dim=1)
    b = torch.cat((response[:, :1], update[:, 1:] * response[:, 1:]), dim=1)
    stride = 1
    while stride < response.shape[1]:
        # Read both old arrays before advancing to the next parallel scan stage.
        next_b = torch.cat((b[:, :stride],
            b[:, stride:] + a[:, stride:] * b[:, :-stride]), dim=1)
        a = torch.cat((a[:, :stride], a[:, stride:] * a[:, :-stride]), dim=1)
        b = next_b
        stride *= 2
    return b


class V8MemoryPressureRNN(V5PressureRNN):
    """Unchanged v005 scalar model plus eight learned control-response memories.

    The original encoder/head parameter names and shapes are retained. The
    residual branch reads only standardized R/C/u_in and known physical dt;
    reconstructing dt uses fitter-only registered scaling buffers. The shared
    scalar output head is untouched. No state is retained between forwards.
    """
    def __init__(self, model_spec, n_features, preprocessing, common_base_state=None):
        super().__init__(model_spec, n_features)
        baseline_state = self.state_dict()
        if self.spec['rc_heads']:
            raise ValueError('v008 memory arm preserves the original scalar head')
        names, indices, mean, std = _v8_memory_context(preprocessing, n_features)
        self.memory_feature_names = names
        self.memory_source_names = V8_MEMORY_SOURCES
        self.register_buffer('memory_feature_indices', torch.as_tensor(indices))
        self.register_buffer('memory_feature_mean', torch.as_tensor(mean))
        self.register_buffer('memory_feature_std', torch.as_tensor(std))
        self.memory_drive = nn.Linear(3, V8_MEMORY_CHANNELS)
        initial_tau = torch.tensor(V8_MEMORY_INITIAL_TAU_SECONDS, dtype=torch.float32)
        # inverse softplus, comfortably inside the nonzero gradient range
        self.memory_raw_tau = nn.Parameter(torch.log(torch.expm1(
            initial_tau - V8_MEMORY_TAU_FLOOR_SECONDS)))
        self.memory_head = nn.Sequential(nn.Linear(V8_MEMORY_CHANNELS, 16), nn.SiLU(),
            nn.Linear(16, 1))
        nn.init.zeros_(self.memory_head[-1].weight)
        nn.init.zeros_(self.memory_head[-1].bias)
        self.load_common_base_state(baseline_state if common_base_state is None
            else common_base_state)

    def load_common_base_state(self, state_dict):
        """Copy the exact common baseline initialization, rejecting partial copies."""
        expected = {name for name in self.state_dict() if not name.startswith('memory_')}
        if set(state_dict) != expected:
            raise ValueError('Common baseline state must contain exactly the original v005 model keys')
        incompatible = self.load_state_dict(state_dict, strict=False)
        if incompatible.unexpected_keys or any(not name.startswith('memory_')
                for name in incompatible.missing_keys):
            raise RuntimeError('Common baseline initialization copy failed')
        return incompatible

    def memory_time_constants(self):
        """Learned positive time constants in seconds, serialized as parameters."""
        return F.softplus(self.memory_raw_tau) + V8_MEMORY_TAU_FLOOR_SECONDS

    def memory_states(self, x):
        if x.ndim != 3 or x.shape[-1] != self.n_features + 1:
            raise ValueError('Expected standardized numeric features plus internal R/C code')
        selected = x.index_select(-1, self.memory_feature_indices)
        # Only R, C, u_in drive the learned nonlinearity. Time only sets decay.
        response = torch.tanh(self.memory_drive(selected[..., :3]))
        dt = selected[..., 3] * self.memory_feature_std[3] + self.memory_feature_mean[3]
        return _v8_response_memory_scan(response, dt, self.memory_time_constants())

    def forward(self, x):
        baseline = super().forward(x)
        return baseline + self.memory_head(self.memory_states(x)).squeeze(-1)


def v8_memory_model(model_spec, n_features, preprocessing, common_base_state=None):
    """Use the fit_preprocessor result (x_mean/x_std/feature_names), never labels."""
    return V8MemoryPressureRNN(model_spec, n_features, preprocessing,
        common_base_state=common_base_state)


In [ ]:
def v8_memory_selfcheck(device='cpu'):
    """Synthetic model/recurrence checks; no data reads, training run, or files.

    Strict deterministic execution is enabled during these checks and the caller's
    RNG/backend settings are restored. The production-size timing/memory and exact
    training-resume gates belong to the common v008 trainer on the actual GPU.
    """
    selected = _sequence_device(device)
    generator = torch.Generator()
    saved_rng = _rng_state(generator)
    backend = dict(threads=torch.get_num_threads(),
        deterministic=torch.are_deterministic_algorithms_enabled(),
        warn_only=torch.is_deterministic_algorithms_warn_only_enabled(),
        benchmark=torch.backends.cudnn.benchmark,
        cudnn_deterministic=torch.backends.cudnn.deterministic,
        matmul_tf32=torch.backends.cuda.matmul.allow_tf32,
        cudnn_tf32=torch.backends.cudnn.allow_tf32)
    checks = {}
    def check(name, condition):
        if not bool(condition):
            raise AssertionError(name)
        checks[name] = True
    try:
        seed_sequence(20261102)
        names = ['R', 'C', 'time_step', 'u_in', 'u_out', 'step', 'dt', 'unrelated']
        means = np.asarray([20., 20., 1., 12., .5, 39.5, .03, -4.], dtype=np.float32)
        scales = np.asarray([15., 10., .8, 9., .5, 23., .005, 3.], dtype=np.float32)
        prep = dict(feature_names=names, x_mean=means, x_std=scales)
        spec = dict(family='lstm', hidden=8, layers=2, dropout=.15,
            bidirectional=True, head_width=7)
        base = V5PressureRNN(spec, len(names)).to(selected)
        model = v8_memory_model(spec, len(names), prep,
            common_base_state=base.state_dict()).to(selected)
        x = torch.randn(3, V5_STEPS, len(names)+1, device=selected)
        x[..., names.index('dt')] = torch.linspace(-.7, .7, V5_STEPS,
            device=selected)[None, :]
        x[..., -1] = torch.tensor([0, 4, 8], device=selected)[:, None]
        base.eval()
        model.eval()
        with torch.no_grad():
            original = model(x)
            states = model.memory_states(x)
            check('zero_residual_baseline_prediction_exact', torch.equal(original, base(x)))
            check('scalar_output_shape_finite', original.shape == (3, V5_STEPS)
                and torch.isfinite(original).all())
            check('eight_bounded_finite_memory_channels', states.shape == (3, V5_STEPS, 8)
                and torch.isfinite(states).all() and states.abs().max() <= 1.000001)
            check('common_baseline_parameters_exact', all(torch.equal(value,
                model.state_dict()[name]) for name, value in base.state_dict().items()))
            single_states = torch.cat([model.memory_states(x[i:i+1]) for i in range(3)])
            check('independent_breath_memory', torch.allclose(states, single_states,
                rtol=1e-6, atol=1e-7))
            model.memory_states(x.flip(0))
            check('repeated_forward_resets_memory', torch.equal(states, model.memory_states(x)))
            check('permuting_breaths_only_permutates_memory', torch.allclose(
                states.flip(0), model.memory_states(x.flip(0)), rtol=1e-6, atol=1e-7))
            changed = x.clone()
            unrelated = [i for i, name in enumerate(names) if name not in V8_MEMORY_SOURCES]
            changed[..., unrelated] += 1000.
            changed[..., -1] = 2.
            check('memory_reads_only_R_C_u_in_dt', torch.equal(states,
                model.memory_states(changed)))
            changed = x.clone()
            changed[:, 37:, names.index('u_in')] += 3.
            check('memory_prefix_causal', torch.equal(states[:, :37],
                model.memory_states(changed)[:, :37]))
            check('positive_initial_time_constants',
                torch.all(model.memory_time_constants() > 0))
            saved_tau = model.memory_raw_tau.detach().clone()
            model.memory_raw_tau.copy_(torch.tensor([-1e4, 1e4] * 4, device=selected))
            check('extreme_time_constants_finite', torch.isfinite(model.memory_states(x)).all()
                and torch.all(model.memory_time_constants() > 0))
            model.memory_raw_tau.copy_(saved_tau)
            # Irregular elapsed-time analytic reference with a zero initial drive.
            response = torch.ones(2, V5_STEPS, 8, device=selected)
            response[:, 0] = 0.
            intervals = .01 + torch.arange(V5_STEPS, device=selected)[None, :] % 7 * .001
            intervals = intervals.expand(2, -1).clone()
            intervals[:, 0] = 0.
            tau = model.memory_time_constants()
            actual = _v8_response_memory_scan(response, intervals, tau)
            # CPU cumulative elapsed time avoids Torch 2.2 CUDA cumsum restrictions.
            elapsed = intervals.detach().cpu().cumsum(1).to(selected)
            expected = -torch.expm1(-elapsed.unsqueeze(-1) / tau)
            check('irregular_elapsed_time_matches_analytic_decay', torch.allclose(
                actual, expected, rtol=1e-5, atol=2e-6))
            faster = _v8_response_memory_scan(response, intervals*2., tau)
            check('increased_elapsed_time_moves_toward_response',
                torch.all(faster >= actual-2e-6))
            check('constant_response_temporally_monotone',
                torch.all(actual[:, 1:] >= actual[:, :-1]-2e-6))
            # Independent direct recurrence checks the parallel scan, including dt=0.
            reference = [response[:, 0]]
            for step in range(1, V5_STEPS):
                exponent = -intervals[:, step:step+1] / tau
                reference.append(torch.exp(exponent)*reference[-1]
                    - torch.expm1(exponent)*response[:, step])
            check('parallel_scan_matches_step_reference', torch.allclose(actual,
                torch.stack(reference, dim=1), rtol=1e-5, atol=2e-6))
        # Varying response oracle also checks gradients through the parallel scan.
        response = torch.randn(3, V5_STEPS, 8, device=selected).tanh().requires_grad_()
        intervals = (.025 + .015*torch.rand(3, V5_STEPS, device=selected)).requires_grad_()
        tau = model.memory_time_constants().detach().requires_grad_()
        parallel = _v8_response_memory_scan(response, intervals, tau)
        reference = [response[:, 0]]
        for step in range(1, V5_STEPS):
            exponent = -intervals[:, step:step+1] / tau
            reference.append(torch.exp(exponent)*reference[-1]
                - torch.expm1(exponent)*response[:, step])
        reference = torch.stack(reference, dim=1)
        weights = torch.randn_like(parallel)
        parallel_gradients = torch.autograd.grad((parallel*weights).sum(),
            (response, intervals, tau), retain_graph=True)
        reference_gradients = torch.autograd.grad((reference*weights).sum(),
            (response, intervals, tau))
        check('varying_response_parallel_scan_matches_loop', torch.allclose(parallel,
            reference, rtol=1e-5, atol=2e-6))
        for label, actual_gradient, expected_gradient in zip(
                ('response', 'elapsed_intervals', 'time_constants'),
                parallel_gradients, reference_gradients):
            check('parallel_scan_'+label+'_gradient_matches_loop', torch.allclose(
                actual_gradient, expected_gradient, rtol=1e-4, atol=3e-5))
        # The first real optimizer update opens the zero residual output; the
        # second gives all upstream memory parameters nonzero finite gradients.
        model.train()
        target = .2*x[..., names.index('u_in')] + .1*torch.sin(x[..., 2]) + .3
        optimizer = torch.optim.Adam(model.parameters(), lr=1e-3, weight_decay=0.)
        memory_parameters = {name: parameter for name, parameter in model.named_parameters()
            if name.startswith('memory_')}
        for update in range(2):
            optimizer.zero_grad(set_to_none=True)
            loss = (model(x)-target).abs().mean()
            loss.backward()
            check('all_parameter_gradients_finite_update_'+str(update+1), all(
                parameter.grad is not None and torch.isfinite(parameter.grad).all()
                for parameter in model.parameters()))
            if update == 0:
                check('residual_output_gradient_nonzero_first_update',
                    model.memory_head[-1].weight.grad.abs().sum() > 0)
                check('upstream_memory_gradient_zero_at_initial_closed_output',
                    torch.count_nonzero(model.memory_drive.weight.grad) == 0
                    and torch.count_nonzero(model.memory_raw_tau.grad) == 0)
            else:
                check('all_memory_parameter_gradients_present_finite_nonzero', all(
                    parameter.grad is not None and torch.isfinite(parameter.grad).all()
                    and parameter.grad.abs().sum() > 0
                    for parameter in memory_parameters.values()))
            nn.utils.clip_grad_norm_(model.parameters(), 1.)
            optimizer.step()
        model.eval()
        with torch.no_grad():
            prediction = model(x)
            clone = v8_memory_model(spec, len(names), prep).to(selected)
            clone.load_state_dict(model.state_dict(), strict=True)
            clone.eval()
            check('strict_state_roundtrip_prediction_exact', torch.equal(prediction, clone(x)))
        checkpoint_keys = set(model.state_dict())
        check('training_scalers_and_indices_are_checkpoint_buffers', all(name in checkpoint_keys
            for name in ('memory_feature_indices', 'memory_feature_mean', 'memory_feature_std')))
        invalid_prep = dict(prep, feature_names=['pressure']+names[1:])
        try:
            v8_memory_model(spec, len(names), invalid_prep)
        except ValueError:
            checks['reject_pressure_feature_name'] = True
        else:
            raise AssertionError('reject_pressure_feature_name')
        return dict(component=V8_MEMORY_VERSION, synthetic_only=True, dataset_reads=0,
            device=str(selected), torch_version=torch.__version__,
            cuda_version=torch.version.cuda, strict_deterministic=True,
            memory_sources=list(V8_MEMORY_SOURCES), channels=V8_MEMORY_CHANNELS,
            parameter_count=int(sum(parameter.numel() for parameter in model.parameters())),
            added_parameters=int(sum(parameter.numel() for parameter in memory_parameters.values())),
            checks=checks, checks_total=len(checks), passed=all(checks.values()))
    finally:
        _restore_rng(saved_rng, generator)
        torch.set_num_threads(backend['threads'])
        torch.use_deterministic_algorithms(backend['deterministic'], warn_only=backend['warn_only'])
        torch.backends.cudnn.benchmark = backend['benchmark']
        torch.backends.cudnn.deterministic = backend['cudnn_deterministic']
        torch.backends.cuda.matmul.allow_tf32 = backend['matmul_tf32']
        torch.backends.cudnn.allow_tf32 = backend['cudnn_tf32']
